# Motor imagery decoding from 59-channel EEG with CP decomposition

BCI Competition IV dataset 1. Tensor is trials x channels x time (200 x 59 x 300).
Rank-4 CP gives 4 spatial maps, 4 time courses, and 4 loadings per trial.
Question: can those 4 loadings per trial decode which movement was imagined?

CP is fit on train folds only, test trials are projected onto the fixed factors, 10-fold CV.

## 1. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorly"], check=True)

In [ ]:
import os, glob, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import butter, filtfilt
import tensorly as tl
from tensorly.decomposition import parafac
from tensorly.tenalg import khatri_rao
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
rng = np.random.default_rng(SEED)
tl.set_backend("numpy")

## 2. Data

Download the 100 Hz calibration .mat files from https://www.bbci.de/competition/iv/ (free sign-up)
and upload them to Colab or Drive.

Subjects c, d, e are artificial, so only using a, b, f, g.

In [ ]:
candidates = [os.environ.get("EEG_DIR", ""), ".", "/content", "/content/drive/MyDrive"]
mat_files = []
for d in candidates:
    if d and os.path.isdir(d):
        mat_files = sorted(glob.glob(os.path.join(d, "BCICIV_calib_ds1*.mat")))
        if mat_files:
            break

if not mat_files:
    raise FileNotFoundError("upload BCICIV_calib_ds1*.mat first")

REAL = {"a", "b", "f", "g"}
mat_files = [f for f in mat_files if os.path.basename(f)[-5] in REAL]
print([os.path.basename(f) for f in mat_files])

## 3. Load one subject

In [ ]:
def load_subject(mat_path):
    m = loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    cnt = 0.1 * m["cnt"].astype(np.float64)   # to microvolts
    pos = np.atleast_1d(m["mrk"].pos).astype(int)
    y = np.atleast_1d(m["mrk"].y).astype(int)
    nfo = m["nfo"]
    info = {
        "fs": int(nfo.fs),
        "clab": [str(c) for c in np.atleast_1d(nfo.clab)],
        "classes": [str(c) for c in np.atleast_1d(nfo.classes)],
        "xpos": np.asarray(nfo.xpos, dtype=float),
        "ypos": np.asarray(nfo.ypos, dtype=float),
    }
    return cnt, pos, y, info

cnt, pos, y, info = load_subject(mat_files[0])
print("cnt:", cnt.shape)
print("fs:", info["fs"], "Hz")
print("cues:", len(pos), "| classes:", info["classes"], "| labels:", dict(zip(*np.unique(y, return_counts=True))))
print(info["clab"][:10])

In [ ]:
# raw EEG, first 5 s, 5 channels
t = np.arange(0, 5 * info["fs"]) / info["fs"]
plt.figure(figsize=(12, 4))
for k in range(5):
    plt.plot(t, cnt[:len(t), k] + 40 * k, lw=0.7, label=info["clab"][k])
plt.xlabel("s"); plt.ylabel("uV (offset)"); plt.legend(loc="right")
plt.show()

## 4. Preprocess

8-30 Hz band-pass (mu + beta), epoch 0.5-3.5 s after cue, standardise each trial.

In [ ]:
def bandpass(data, fs, low=8.0, high=30.0, order=4):
    b, a = butter(order, [low / (fs / 2), high / (fs / 2)], btype="band")
    return filtfilt(b, a, data, axis=0)

def make_trials(cnt, pos, fs, t_start=0.5, t_end=3.5):
    s0, s1 = int(t_start * fs), int(t_end * fs)
    return np.stack([cnt[p + s0:p + s1].T for p in pos])   # (trials, channels, time)

cnt_f = bandpass(cnt, info["fs"])
X = make_trials(cnt_f, pos, info["fs"])
X = (X - X.mean(axis=(1, 2), keepdims=True)) / X.std(axis=(1, 2), keepdims=True)
labels = (y > 0).astype(int)
print("X:", X.shape)

## 5. CP decomposition

X[trial, ch, t] ~ sum_r C[trial, r] * A[ch, r] * B[t, r]

A = spatial maps, B = time courses, C = trial loadings (the features).

In [ ]:
RANK = 4

def fit_cp(X_train, rank=RANK, seed=SEED):
    weights, (C, A, B) = parafac(tl.tensor(X_train), rank=rank, init="random",
                                 random_state=seed, n_iter_max=300, tol=1e-7, normalize_factors=True)
    return A, B, C * weights

A, B, C = fit_cp(X)
print("A:", A.shape, "B:", B.shape, "C:", C.shape)

recon = tl.cp_to_tensor((np.ones(RANK), [C, A, B]))
explained = 1 - np.sum((X - recon) ** 2) / np.sum(X ** 2)
print(f"variance explained: {explained*100:.1f}%")

In [ ]:
def plot_components(A, B, info, fs, title=""):
    R = A.shape[1]
    fig, axes = plt.subplots(2, R, figsize=(3.2 * R, 6))
    for r in range(R):
        ax = axes[0, r]
        v = np.abs(A[:, r]).max()
        ax.scatter(info["xpos"], info["ypos"], c=A[:, r], cmap="RdBu_r", vmin=-v, vmax=v, s=60)
        ax.set_title(f"comp {r+1}"); ax.set_aspect("equal"); ax.axis("off")
        ax = axes[1, r]
        ax.plot(np.arange(B.shape[0]) / fs + 0.5, B[:, r], lw=0.8)
        ax.set_xlabel("s after cue")
    fig.suptitle(title); plt.tight_layout(); plt.show()

plot_components(A, B, info, info["fs"], title=f"subject {os.path.basename(mat_files[0])[-5]}")

## 6. Decoding (10-fold CV)

Fit CP on train folds, project test trials onto A and B by least squares, LDA on the loadings.
Baseline: log band power per channel + LDA.

In [ ]:
def project_trials(X_new, A, B):
    KR = khatri_rao([B, A])
    pinv = np.linalg.pinv(KR)
    V = X_new.reshape(X_new.shape[0], -1, order="F")
    return V @ pinv.T

def cv_decode(X, labels, rank=RANK, n_splits=10, seed=SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    cp_acc, bp_acc = [], []
    for tr, te in skf.split(X, labels):
        A, B, C_tr = fit_cp(X[tr], rank=rank, seed=seed)
        C_te = project_trials(X[te], A, B)
        clf = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())
        clf.fit(C_tr, labels[tr]);  cp_acc.append(clf.score(C_te, labels[te]))

        P_tr = np.log(X[tr].var(axis=2));  P_te = np.log(X[te].var(axis=2))
        clf2 = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
        clf2.fit(P_tr, labels[tr]);  bp_acc.append(clf2.score(P_te, labels[te]))
    return np.mean(cp_acc), np.std(cp_acc), np.mean(bp_acc), np.std(bp_acc)

cp_m, cp_s, bp_m, bp_s = cv_decode(X, labels)
print(f"subject {os.path.basename(mat_files[0])[-5]}")
print(f"  CP rank-{RANK} + LDA : {cp_m*100:5.1f}% (+/- {cp_s*100:.1f})")
print(f"  band-power + LDA   : {bp_m*100:5.1f}% (+/- {bp_s*100:.1f})")
print(f"  chance             : 50.0%")

## 7. All four subjects

In [ ]:
per_subject = {}
for f in mat_files:
    sid = os.path.basename(f)[-5]
    cnt, pos, y, info = load_subject(f)
    Xs = make_trials(bandpass(cnt, info["fs"]), pos, info["fs"])
    Xs = (Xs - Xs.mean(axis=(1, 2), keepdims=True)) / Xs.std(axis=(1, 2), keepdims=True)
    ls = (y > 0).astype(int)
    cp_m, cp_s, bp_m, bp_s = cv_decode(Xs, ls)
    per_subject[sid] = {"classes": info["classes"], "cp_acc": cp_m, "cp_std": cp_s, "bp_acc": bp_m, "bp_std": bp_s}
    print(f"subject {sid} ({' vs '.join(info['classes'])}): CP {cp_m*100:5.1f}% | band-power {bp_m*100:5.1f}%")

mean_cp = np.mean([v["cp_acc"] for v in per_subject.values()])
mean_bp = np.mean([v["bp_acc"] for v in per_subject.values()])
print(f"\nmean: CP {mean_cp*100:.1f}% | band-power {mean_bp*100:.1f}%")

In [ ]:
sids = list(per_subject)
xpos_ = np.arange(len(sids))
plt.figure(figsize=(7, 4))
plt.bar(xpos_ - 0.2, [per_subject[s]["cp_acc"] * 100 for s in sids], 0.4, label=f"CP rank-{RANK} + LDA",
        yerr=[per_subject[s]["cp_std"] * 100 for s in sids], capsize=3)
plt.bar(xpos_ + 0.2, [per_subject[s]["bp_acc"] * 100 for s in sids], 0.4, label="band-power + LDA",
        yerr=[per_subject[s]["bp_std"] * 100 for s in sids], capsize=3)
plt.axhline(50, color="gray", ls="--", lw=1, label="chance")
plt.xticks(xpos_, [f"subject {s}" for s in sids]); plt.ylabel("10-fold CV acc (%)"); plt.ylim(0, 100)
plt.legend(); plt.show()

## 8. Rank sweep

In [ ]:
cnt, pos, y, info = load_subject(mat_files[0])
X = make_trials(bandpass(cnt, info["fs"]), pos, info["fs"])
X = (X - X.mean(axis=(1, 2), keepdims=True)) / X.std(axis=(1, 2), keepdims=True)
labels = (y > 0).astype(int)

ranks = [1, 2, 3, 4, 6, 8]
sweep = []
for r in ranks:
    m, s, _, _ = cv_decode(X, labels, rank=r, n_splits=5)
    sweep.append(m); print(f"rank {r}: {m*100:5.1f}%")
plt.figure(figsize=(6, 4)); plt.plot(ranks, np.array(sweep) * 100, marker="o")
plt.axhline(50, color="gray", ls="--", lw=1); plt.xlabel("rank"); plt.ylabel("acc (%)")
plt.show()

## 9. Save results

In [ ]:
summary = {
    "dataset": "BCI Competition IV, data set 1, 100 Hz calibration, subjects a/b/f/g",
    "channels": 59, "band_hz": [8, 30], "window_s": [0.5, 3.5], "cp_rank": RANK,
    "per_subject": {s: {k: (round(float(v), 3) if isinstance(v, (float, np.floating)) else v)
                        for k, v in d.items()} for s, d in per_subject.items()},
    "mean_cp_accuracy_pct": round(mean_cp * 100, 1),
    "mean_bandpower_accuracy_pct": round(mean_bp * 100, 1),
    "rank_sweep_subject_a": dict(zip(ranks, [round(v * 100, 1) for v in sweep])),
}
with open("results_eeg_cp.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))